<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-04-rag/lesson-4.3-rag-engine/notebooks/GCP_Capstone_4.3_RAG_Engine.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4.3 Vertex AI RAG Engine — Managed RAG Pipeline
**Netsetos GenAI Engineering — GCP Capstone**

Create corpus, import from GCS/Drive/Slack/Jira, retrieve, and generate with automatic citations.


## Setup


In [ ]:
!pip install -q google-cloud-aiplatform google-genai
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE

from vertexai import rag
import vertexai
from google import genai
from google.genai import types

vertexai.init(project=PROJECT_ID, location='us-central1')
client = genai.Client(enterprise=True, project=PROJECT_ID, location='global')  # Gemini 3.x generation: global (corpus stays regional)


## Prepare a new project for serverless RAG Engine (one-time)
A brand-new project needs three things before `create_corpus` works, all handled by the cell below (idempotent — safe to re-run):
1. **APIs enabled** — `vectorsearch.googleapis.com` backs the serverless vector store and is **off by default**; a missing enable is exactly what makes `create_corpus` fail with *“Vector Search API … has not been used … or it is disabled”*. (`aiplatform` + `storage` are enabled too.)
2. **Serverless mode** — the default “Scaled”/Spanner store is allowlist-only for new projects; Serverless mode is open to everyone.
3. **Propagation** — API enablement is eventually consistent, so the first corpus is created through `create_corpus_ready(...)`, which retries while enablement propagates.

> Serverless RAG corpora (Vector Search 2.0) are **`us-central1`-only** — not `asia-south1`. The corpus, embedding and retrieval client stays regional; only the Gemini-3.x generation client uses `global`.

In [ ]:
# One-time new-project prep for serverless RAG Engine. Idempotent — safe to re-run.
import google.auth, google.auth.transport.requests, requests, subprocess, time
from google.api_core import exceptions as _gexc
_RAG_LOCATION = 'us-central1'   # serverless RAG corpora (Vector Search 2.0) are us-central1-only

# 1) Enable the APIs a fresh project needs. vectorsearch backs the serverless vector store
#    (off by default -> the create_corpus 403); storage is for GCS import.
subprocess.run(['gcloud', 'services', 'enable',
                'aiplatform.googleapis.com', 'vectorsearch.googleapis.com',
                'storage.googleapis.com', '--project', PROJECT_ID], check=False)

# 2) Switch RAG Engine to Serverless mode (open to everyone, no allowlist).
_creds, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/cloud-platform'])
_creds.refresh(google.auth.transport.requests.Request())
_r = requests.patch(
    f'https://{_RAG_LOCATION}-aiplatform.googleapis.com/v1beta1/'
    f'projects/{PROJECT_ID}/locations/{_RAG_LOCATION}/ragEngineConfig',
    headers={'Authorization': f'Bearer {_creds.token}'},
    json={'ragManagedDbConfig': {'serverless': {}}}, timeout=60)
print('Serverless mode:', 'ready' if _r.ok else f'{_r.status_code} {_r.text[:150]}')

# 3) create_corpus wrapper that waits out API-enablement propagation on fresh projects.
def create_corpus_ready(**kwargs):
    for _attempt in range(8):  # ~ up to 4 min
        try:
            return rag.create_corpus(**kwargs)
        except (_gexc.PermissionDenied, _gexc.FailedPrecondition) as e:
            m = str(e).lower()
            if 'has not been used' in m or 'is disabled' in m or 'service_disabled' in m:
                print('vectorsearch API still propagating; retrying in 30s...'); time.sleep(30); continue
            raise
    raise RuntimeError('vectorsearch.googleapis.com still not ready — wait 1-2 min and re-run.')
print('APIs enabled. First corpus uses create_corpus_ready(...) to ride out enablement propagation.')

## Create a docs bucket (named from your project)
The lesson imports documents from Cloud Storage. This creates a bucket `{PROJECT_ID}-rag-docs` in `us-central1` and seeds a sample file, so `import_files` has real content to ingest. Idempotent — safe to re-run.

In [ ]:
# Create a GCS bucket for this lesson's documents (named from the project so it is
# globally unique) and seed a sample file. Idempotent — safe to re-run.
import subprocess
BUCKET = f'{PROJECT_ID}-rag-docs'
_exists = subprocess.run(['gcloud', 'storage', 'buckets', 'describe', f'gs://{BUCKET}',
                          '--project', PROJECT_ID], capture_output=True, text=True).returncode == 0
if not _exists:
    subprocess.run(['gcloud', 'storage', 'buckets', 'create', f'gs://{BUCKET}',
                    '--project', PROJECT_ID, '--location', 'us-central1',
                    '--uniform-bucket-level-access'], check=True)
_sample = ('DocuMind RAG sample document.\n'
           'Retrieval-Augmented Generation (RAG) grounds an LLM in your own documents: it chunks\n'
           'documents, embeds the chunks, stores the vectors, retrieves the nearest chunks for a\n'
           'query, and passes them to the model as context so answers cite real sources.\n'
           'Vertex AI RAG Engine manages chunking, embedding, indexing and retrieval for you.\n')
with open('sample.txt', 'w') as _f:
    _f.write(_sample)
subprocess.run(['gcloud', 'storage', 'cp', 'sample.txt', f'gs://{BUCKET}/docs/sample.txt',
                '--project', PROJECT_ID], check=True)
print(f'Docs bucket ready: gs://{BUCKET}/docs/  (seeded sample.txt)')

## Cell 1: Create a RAG Corpus


In [ ]:
embedding_config = rag.RagEmbeddingModelConfig(
    vertex_prediction_endpoint=rag.VertexPredictionEndpoint(
        publisher_model='publishers/google/models/text-embedding-005'
    )
)

corpus = create_corpus_ready(
    display_name='documind-lesson43',
    description='Lesson 4.3 test corpus',
    backend_config=rag.RagVectorDbConfig(
        rag_embedding_model_config=embedding_config),
)
print(f'Corpus: {corpus.name}')

# List corpora
for c in rag.list_corpora():
    print(f'  {c.display_name}: {c.name}')


## Cell 2: Import from GCS


In [ ]:
# Grant the RAG service agent read on the docs bucket (create_corpus above provisioned
# it); without this grant, import returns 0 files silently.
import subprocess
_pn = subprocess.run(['gcloud', 'projects', 'describe', PROJECT_ID, '--format=value(projectNumber)'],
                     capture_output=True, text=True).stdout.strip()
subprocess.run(['gcloud', 'storage', 'buckets', 'add-iam-policy-binding', f'gs://{BUCKET}',
                '--member', f'serviceAccount:service-{_pn}@gcp-sa-vertex-rag.iam.gserviceaccount.com',
                '--role', 'roles/storage.objectViewer', '--project', PROJECT_ID], check=False)

response = rag.import_files(
    corpus.name,
    [f'gs://{BUCKET}/docs/'],
    transformation_config=rag.TransformationConfig(
        chunking_config=rag.ChunkingConfig(
            chunk_size=512, chunk_overlap=100)),
    max_embedding_requests_per_min=900,
)
print(f'Imported: {response.imported_rag_files_count}')
print(f'Skipped: {response.skipped_rag_files_count}')
if response.imported_rag_files_count == 0:
    print('WARNING: 0 files imported. The service-agent grant above may still be '
          'propagating (~1-2 min) — wait and re-run this cell.')

## Cell 3: Import from Google Drive


In [ ]:
# OPTIONAL — import from a Google Drive folder (Cell 2's GCS path already showed ingestion).
# Drive folders are your own, so this one is bring-your-own:
#   1. Put files in a Drive folder; copy its id from the URL
#      (drive.google.com/drive/folders/<THIS_PART>).
#   2. Share that folder as Viewer with the RAG service agent printed below.
#   3. Set DRIVE_FOLDER_ID and re-run.
import subprocess
_pn = subprocess.run(['gcloud', 'projects', 'describe', PROJECT_ID, '--format=value(projectNumber)'],
                     capture_output=True, text=True).stdout.strip()
print(f'Share your Drive folder (Viewer) with: service-{_pn}@gcp-sa-vertex-rag.iam.gserviceaccount.com')

DRIVE_FOLDER_ID = 'YOUR_FOLDER_ID'  # CHANGE, or leave as-is to skip
if DRIVE_FOLDER_ID == 'YOUR_FOLDER_ID':
    print('Skipping Drive import — set DRIVE_FOLDER_ID to your own folder id (shared above) to run.')
else:
    drive_response = rag.import_files(
        corpus.name,
        [f'https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}'],
        transformation_config=rag.TransformationConfig(
            chunking_config=rag.ChunkingConfig(chunk_size=512, chunk_overlap=100)),
        max_embedding_requests_per_min=900)
    print(f'Drive import: {drive_response.imported_rag_files_count}')

## Cell 4: Direct Retrieval


In [ ]:
response = rag.retrieval_query(
    rag_resources=[rag.RagResource(rag_corpus=corpus.name)],
    text='What is RAG?',
    rag_retrieval_config=rag.RagRetrievalConfig(
        top_k=5,
        filter=rag.Filter(vector_distance_threshold=0.5)),
)

for ctx in response.contexts.contexts:
    print(f'Source: {ctx.source_uri}')
    print(f'Score: {ctx.score:.3f}  (cosine distance, lower = more relevant)')
    print(f'Text: {ctx.text[:150]}...\n')


## Cell 5: Grounded Generation


In [ ]:
rag_retrieval_tool = types.Tool(
    retrieval=types.Retrieval(
        vertex_rag_store=types.VertexRagStore(
            rag_resources=[types.VertexRagStoreRagResource(rag_corpus=corpus.name)],
            rag_retrieval_config=types.RagRetrievalConfig(
                top_k=5,
                filter=types.RagRetrievalConfigFilter(vector_distance_threshold=0.5)))))

response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='Summarize the main topics in my documents',
    config=types.GenerateContentConfig(tools=[rag_retrieval_tool]))
print(response.text)

# Grounding metadata — automatic citations
for candidate in response.candidates:
    gm = candidate.grounding_metadata
    if gm and gm.grounding_chunks:
        for chunk in gm.grounding_chunks:
            print(f'  Source: {chunk.retrieved_context.uri}')
    if gm and gm.grounding_supports:
        for support in gm.grounding_supports:
            print(f'  Claim: {support.segment.text}')
            print(f'  Backed by: {support.grounding_chunk_indices}')


## Cell 6: ManagedRAG Module


In [ ]:
class ManagedRAG:
    def __init__(self, project, location='us-central1'):
        vertexai.init(project=project, location=location)
        self.client = genai.Client(enterprise=True, project=project, location='global')  # generation: global (corpus stays regional)
        self.corpus = None

    def create_corpus(self, name, description=''):
        emb = rag.RagEmbeddingModelConfig(
            vertex_prediction_endpoint=rag.VertexPredictionEndpoint(
                publisher_model='publishers/google/models/text-embedding-005'))
        self.corpus = rag.create_corpus(
            display_name=name, description=description,
            backend_config=rag.RagVectorDbConfig(rag_embedding_model_config=emb))
        return self.corpus.name

    def use_corpus(self, corpus_name):
        self.corpus = rag.get_corpus(name=corpus_name)

    def ingest(self, paths, chunk_size=512, chunk_overlap=100):
        return rag.import_files(
            self.corpus.name, paths,
            transformation_config=rag.TransformationConfig(
                rag.ChunkingConfig(chunk_size=chunk_size, chunk_overlap=chunk_overlap)),
            max_embedding_requests_per_min=900)

    def retrieve(self, query, top_k=5):
        return rag.retrieval_query(
            rag_resources=[rag.RagResource(rag_corpus=self.corpus.name)],
            text=query,
            rag_retrieval_config=rag.RagRetrievalConfig(
                top_k=top_k, filter=rag.Filter(vector_distance_threshold=0.5)))

    def ask(self, question, model_name='gemini-3.6-flash'):
        rag_tool = types.Tool(retrieval=types.Retrieval(
            vertex_rag_store=types.VertexRagStore(
                rag_resources=[types.VertexRagStoreRagResource(rag_corpus=self.corpus.name)],
                rag_retrieval_config=types.RagRetrievalConfig(
                    top_k=5, filter=types.RagRetrievalConfigFilter(vector_distance_threshold=0.5)))))
        return self.client.models.generate_content(
            model=model_name, contents=question,
            config=types.GenerateContentConfig(tools=[rag_tool]))

print('ManagedRAG class ready')


## Cell 7: Cleanup


In [ ]:
# Delete corpus when done (saves storage costs)
# rag.delete_corpus(name=corpus.name)
# print('Corpus deleted')


## ✅ Lesson 4.3 Complete!

- ✅ Created RAG corpus with text-embedding-005
- ✅ Imported from GCS (and optionally Drive)
- ✅ Automatic chunking + embedding + indexing
- ✅ Direct retrieval with retrieval_query()
- ✅ Grounded generation with automatic citations
- ✅ ManagedRAG production module

**Module 4 Complete! Three RAG approaches built:**
- 4.1: Document AI ingestion (OCR/Layout/Form)
- 4.2: DIY RAG pipeline (manual embed→retrieve→generate)
- 4.3: Managed RAG Engine (corpus→import→query)

**Next: Module 5 — BigQuery ML & SQL-Native AI**
